In [ ]:
import os
from google.colab import drive
drive.mount('')
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import random

import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torchvision.transforms.functional as TF
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR

from comet_ml import Experiment
import glob

from PIL import Image
from torch.utils.data import Dataset, DataLoader



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("GPU disponível:", torch.cuda.is_available())
print("Nome da GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Nenhuma")
print("Versão do PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "Please enable GPU from runtime settings"


experiment = Experiment(
    api_key="",
    project_name="",
    workspace=""
)

# other packages
import matplotlib.pyplot as plt
import numpy as np
import random
from tqdm.auto import tqdm

In [ ]:



#Herda Dataset do PyTorch
class BrainMRIDataset(Dataset):
    def __init__(self, root_dir, augment=False):
        #Salva o caminho da pasta principal e se vai aplicar augmentação
        #self é substítuido pelo objeto que está sendo criado
        self.root_dir = root_dir
        self.augment = augment  # booleano no lugar de transform — mais seguro
        
        all_files = glob.glob(os.path.join(root_dir, "**/*.tif"), recursive=True)
        self.images_paths = sorted([f for f in all_files if "_mask.tif" not in f])

    def __len__(self):
        return len(self.images_paths)

    def __getitem__(self, idx):
        img_path  = self.images_paths[idx]
        mask_path = img_path.replace(".tif", "_mask.tif")

        #converte para escala de cinza
        image = Image.open(img_path).convert("L")
        mask  = Image.open(mask_path).convert("L")

        # FIX BUG 7: NEAREST preserva os valores binários da máscara
        # Bilinear criaria valores intermediários (0.3, 0.7...) nas bordas do tumor
        # Usamos 256 para acelerar o treinamento e necessitar de menos memória
        image = TF.resize(image, (256, 256))
        mask  = TF.resize(mask,  (256, 256),
                          interpolation=transforms.InterpolationMode.NEAREST)

        # FIX BUG 1: sorteamos os parâmetros UMA vez e aplicamos nos dois
        # Antes: self.transform(image) e self.transform(mask) faziam sorteios separados
        if self.augment:
            # Sorteia uma vez — True ou False para os dois
            if random.random() > 0.5:
                image = TF.hflip(image)
                mask  = TF.hflip(mask)

            # Sorteia o ângulo uma vez — o mesmo ângulo para os dois
            angle = transforms.RandomRotation.get_params([-10, 10])
            image = TF.rotate(image, angle)
            mask  = TF.rotate(mask,  angle)

        image = TF.to_tensor(image)
        mask  = TF.to_tensor(mask)

        # FIX BUG 6: normalização só na imagem (ajuda a aprender mais rápido, e evita problemas como gradientes explodindo) — a máscara não deve ser normalizada
        # Antes: self.transform(mask) aplicava Normalize junto com o resto
        image = TF.normalize(image, mean=[0.5], std=[0.5])

        mask = (mask > 0.5).long().squeeze(0)

        return image, mask


# FIX BUG 3: dois objetos separados na memória — cada um com seu próprio __getitem__
# Antes: random_split criava dois Subsets apontando para o mesmo dataset_full,
# então alterar o transform de um alterava o do outro também
train_full = BrainMRIDataset(root_dir='', augment=True)
test_full  = BrainMRIDataset(root_dir='', augment=False)
# Geramos os índices uma vez só e usamos nos dois objetos
# O manual_seed garante que a divisão seja reprodutível entre execuções,
# e que o mesmo paciente nunca apareça nos dois conjuntos
# 1. Calculamos o tamanho apenas do conjunto de treino
total      = len(train_full)
train_size = int(0.8 * total)
val_size   = total - train_size # Mudei o nome para val_size

# 2. Geramos a lista de índices embaralhados
generator = torch.Generator().manual_seed(42)
perm      = torch.randperm(total, generator=generator).tolist()

# 3. CORREÇÃO: Ambos os subsets são tirados do train_full!
# Os primeiros 80% dos índices vão para treino, os últimos 20% vão para validação.
train_dataset = torch.utils.data.Subset(train_full, perm[:train_size])
val_dataset   = torch.utils.data.Subset(train_full, perm[train_size:])

# (O assert foi removido porque agora eles vêm do mesmo dataset base)

# 4. Criamos os DataLoaders (Note que renomeei para val_loader)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# 5. E o seu test_full verdadeiro fica intacto num loader separado!
# Você vai usar esse cara aqui só no final de tudo.
test_loader  = DataLoader(test_full, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x):
        return self.double_conv(x)

      

        # TODO: Define the Linear layer that outputs the classification
        # logits over class labels. Remember that CrossEntropyLoss operates over logits.
class UNET(nn.Module):
    def __init__(self):
        super(UNET, self).__init__()
        # Define the layers of the UNET architecture here
        self.pool = nn.MaxPool2d(2, 2)

        self.down1 = DoubleConv(1, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)
        self.down5 = DoubleConv(512, 1024)
        
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(1024, 512)
        
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(512, 256)
        
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(256, 128)
        
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(128, 64)

        
        self.final_conv = nn.Conv2d(64, 2, kernel_size=1) # Output layer for binary segmentation
        
    def forward(self, x):
        # Downsampling path
        d1 = self.down1(x) # (B, 64, H, W)
        d2 = self.down2(self.pool(d1)) # (B, 128, H/2, W/2)
        d3 = self.down3(self.pool(d2)) # (B, 256, H/4, W/4)
        d4 = self.down4(self.pool(d3)) # (B, 512, H/8, W/8)
        d5 = self.down5(self.pool(d4)) # (B, 1024, H/16, W/16)
        
        # Upsampling path
        u1 = self.up1(d5) # (B, 512, H/8, W/8)
        u1 = torch.cat([u1, d4], dim=1) # Skip connection
        u1 = self.conv_up1(u1) # (B, 512, H/8, W/8)
        
        u2 = self.up2(u1) # (B, 256, H/4, W/4)
        u2 = torch.cat([u2, d3], dim=1) # Skip connection
        u2 = self.conv_up2(u2) # (B, 256, H/4, W/4)

        u3 = self.up3(u2) # (B, 128, H/2, W/2)
        u3 = torch.cat([u3, d2], dim=1) # Skip connection
        u3 = self.conv_up3(u3) # (B, 128, H/2, W/2)

        u4 = self.up4(u3) # (B, 64, H, W)
        u4 = torch.cat([u4, d1], dim=1) # Skip connection
        u4 = self.conv_up4(u4) # (B, 64, H, W)
        output = self.final_conv(u4) # (B, 2, H, W)
        return output

# Instantiate the model

# Initialize the model by passing some data through


In [ ]:


#N vou usar pq a memória da GPU não aguenta
def dice_loss(pred_logits, target, smooth=1e-6):
    """pred_logits: (B, 2, H, W); target: (B, H, W) long"""
    pred_prob = F.softmax(pred_logits, dim=1)[:, 1]  # prob do canal tumor
    target_f  = target.float()
    intersection = (pred_prob * target_f).sum(dim=(1, 2))
    union        = pred_prob.sum(dim=(1, 2)) + target_f.sum(dim=(1, 2))
    dice         = (2. * intersection + smooth) / (union + smooth)
    return 1. - dice.mean()
def combined_loss(pred, target, alpha=0.5):
    ce   = F.cross_entropy(pred, target)
    dice = dice_loss(pred, target)
    return alpha * ce + (1. - alpha) * dice


print(f"Iniciando treinamento no dispositivo: {device}")

# Instancia a sua rede UNET e envia para a memória do dispositivo
#model = UNET().to(device)

# Define a função de perda adequada para classificação de pixels em múltiplas classes
#weights   = compute_class_weights(test_full, device) #mudei o test_full para train_full
num_epochs = 5
criterion = combined_loss
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

# NOVO: A função train agora recebe o val_loader, o experiment e o scheduler
def train(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs, experiment):
    
    melhor_dice = 0.0 # NOVO: Variável para rastrear o melhor modelo
    #scaler = torch.amp.GradScaler('cuda')
    
    for epoch in range(epochs):
        # ==========================================
        #               FASE DE TREINO
        # ==========================================
        model.train() 
        running_loss = 0.0
        n_processed = 0
        
        prog_bar = tqdm(train_loader, desc=f"Época {epoch+1}/{epochs} [Treino]", leave=True)

        for batch_idx, (images, masks) in enumerate(prog_bar):
            images = images.to(device)
            masks = masks.to(device) 
            optimizer.zero_grad()
            outputs = model(images)    
            loss = criterion(outputs, masks)

            #with torch.amp.autocast('cuda'):
                #outputs = model(images)    
                #loss = criterion(outputs, masks)
        
        # 2. Faz o backward pass usando o scaler
            #scaler.scale(loss).backward()
            #scaler.step(optimizer)
            #scaler.update()
            
            
            loss.backward()
            optimizer.step()
            # REMOVIDO: scheduler.step() saiu daqui
            
            running_loss += loss.item() * images.size(0)
            n_processed += images.size(0)
            
            prog_bar.set_postfix(loss_lote=f"{loss.item():.4f}")
            
        epoch_loss = running_loss / n_processed
        
        # NOVO: Atualizamos o scheduler apenas uma vez no fim da época!
        scheduler.step()
        
        # NOVO: Enviando a Loss de Treino para o Comet
        experiment.log_metric("train_loss", epoch_loss, epoch=epoch)
        
        # ==========================================
        #             FASE DE VALIDAÇÃO
        # ==========================================
        model.eval() # Modo de avaliação (desliga Dropout, BatchNorm, etc)
        val_dice_total = 0.0
        n_val_samples  = 0
        
        
        # torch.no_grad() economiza MUITA memória na GPU, pois não calcula gradientes
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f"Época {epoch+1}/{epochs} [Validação]", leave=True)
            for images, masks in val_bar:
                images = images.to(device)
                masks = masks.to(device)
                
                outputs = model(images)
                
                # NOVO: Calculando o Dice. 
                # Como a sua dice_loss retorna (1 - dice), podemos reverter a conta facilmente:
                loss_dice_val = dice_loss(outputs, masks)
                soft_dice_val = 1.0 - loss_dice_val.item() 
                
                val_dice_total += soft_dice_val * images.size(0)  # pondera pelo nº real de amostras
                n_val_samples  += images.size(0)

                
        # Calculando a média do Dice na Validação
        epoch_val_dice = val_dice_total / n_val_samples
        
        print(f"Fim da Época [{epoch+1}/{epochs}] - Loss Treino: {epoch_loss:.4f} | Dice Validação: {epoch_val_dice:.4f}\n")
        
        # NOVO: Enviando o Dice de Validação para o Comet
        experiment.log_metric("val_dice", epoch_val_dice, epoch=epoch)
        
        # NOVO: Salvando o modelo na época em que o Dice for maximizado!
        if epoch_val_dice > melhor_dice:
            melhor_dice = epoch_val_dice
            
            caminho_no_drive = ''

            torch.save(model.state_dict(), caminho_no_drive)
            experiment.log_metric("melhor_val_dice", melhor_dice, epoch=epoch)
            print(f"🌟 Novo melhor modelo salvo! {epoch} 🌟 \n")

In [ ]:
train(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, experiment=experiment)

In [ ]:
caminho_no_drive = ''

torch.save(model.state_dict(), caminho_no_drive)
print(f"🎉 Modelo salvo com sucesso em: {caminho_no_drive}")

In [ ]:
checkpoint = {
    'epoch': 30, # mude para a época em que parou
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict()
}

# Salve esse dicionário completo no Drive
torch.save(checkpoint, '')
print("🎉 Checkpoint completo salvo com sucesso!")

In [ ]:
def dice_score(pred, target, smooth=1e-6):
    """Coeficiente de Dice para segmentação binária."""
    pred_flat = pred.view(-1).float()
    target_flat = target.view(-1).float()
    intersection = (pred_flat * target_flat).sum()
    return (2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)

def iou_score(pred, target, smooth=1e-6):
    """Intersection over Union (Jaccard Index)."""
    pred_flat = pred.view(-1).float()
    target_flat = target.view(-1).float()
    intersection = (pred_flat * target_flat).sum()
    union = pred_flat.sum() + target_flat.sum() - intersection
    return (intersection + smooth) / (union + smooth)

def evaluate(model, dataloader, loss_function, smooth=1e-6):
    model.eval()
    test_loss = 0.0
    total_samples = 0
    
    # Variáveis para acumular apenas os componentes matemáticos (escalares simples)
    total_intersection = 0.0
    total_pred_sum = 0.0
    total_target_sum = 0.0

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Avaliando"):
            images, labels = images.to(device), labels.to(device)
            
            # 1. Avanço do modelo
            outputs = model(images)
            
            # 2. Cálculo da perda ponderada
            batch_n = images.size(0)
            test_loss += loss_function(outputs, labels).item() * batch_n
            total_samples += batch_n
            
            # 3. Extração das predições
            predicted = torch.argmax(outputs, dim=1)
            
            # Achatamos os tensores do lote atual para fazer a matemática pixel a pixel
            pred_flat = predicted.view(-1).float()
            labels_flat = labels.view(-1).float()
            
            # Acumulamos apenas os resultados numéricos (.item() extrai o número puro do tensor)
            total_intersection += (pred_flat * labels_flat).sum().item()
            total_pred_sum += pred_flat.sum().item()
            total_target_sum += labels_flat.sum().item()

    # 4. Cálculo final das médias globais baseado nos totais acumulados
    avg_loss = test_loss / total_samples  
    
    # Fórmula do Dice Global
    global_dice = (2. * total_intersection + smooth) / (total_pred_sum + total_target_sum + smooth)
    
    # Fórmula do IoU Global (União = Soma das partes - Interseção)
    global_iou = (total_intersection + smooth) / (total_pred_sum + total_target_sum - total_intersection + smooth)

    return avg_loss, global_dice, global_iou
# Chama a função corrigida
avg_batch_loss, batch_dice, batch_iou = evaluate(model, test_loader, criterion)
 
# Esse print aqui vai funcionar perfeitamente agora:
print(f"Loss: {avg_batch_loss:.4f}, Dice: {batch_dice:.4f}, IoU: {batch_iou:.4f}")
# CORREÇÃO: Adicionar métricas de segmentação adequadas (Dice Coefficient e IoU)

# Dentro do loop de avaliação:


In [ ]:
import matplotlib.pyplot as plt

def visualize_segmentation(model, dataset, idx=0, device='cuda', alpha=0.4):
    # 1. Pega a imagem diretamente pelo índice do dataset
    image, label = dataset[idx]
    image = image.unsqueeze(0).to(device)  # [1, C, H, W]

    # 2. Forward pass (Predição)
    model.eval()
    with torch.no_grad():
        output = model(image)  # [1, C, H, W]

    # 3. Extrai as máscaras (Predita e Real)
    predicted_mask = torch.argmax(output[0], dim=0).cpu().numpy()  # [H, W]
    label_np       = label.numpy()                                  # [H, W]

    # 4. Prepara a imagem original para exibição
    img = image[0].cpu()
    if img.shape[0] == 1: # Se for Ressonância (Tons de cinza)
        image_np = img.squeeze(0).numpy()        
        # Cria uma versão com 3 canais (RGB) copiando o cinza para manter o formato
        img_rgb = np.stack([image_np, image_np, image_np], axis=-1)
        cmap_img = "gray"
    else: # Se a imagem já for RGB
        image_np = img.permute(1, 2, 0).numpy()  
        img_rgb = image_np.copy()
        cmap_img = None

    # Normaliza as imagens base para o intervalo [0, 1]
    denom_rgb = img_rgb.max() - img_rgb.min()
    img_rgb = (img_rgb - img_rgb.min()) / (denom_rgb if denom_rgb > 0 else 1.0)
    
    if cmap_img == "gray":
        denom_img = image_np.max() - image_np.min()
        image_np = (image_np - image_np.min()) / (denom_img if denom_img > 0 else 1.0)

    # ────────────────────────────────────────────────────────────────────────
    # 5. MÁGICA DA SOBREPOSIÇÃO (FUSÃO DE PIXELS)
    # ────────────────────────────────────────────────────────────────────────
    # Criamos uma cópia da imagem RGB da ressonância
    overlay_rgb = img_rgb.copy()
    
    # Encontramos as coordenadas de onde o modelo detectou o tumor
    tumor_pixels = predicted_mask > 0
    
    # Aplicamos a cor verde DIRETAMENTE nos pixels da imagem, misturando 
    # a intensidade da ressonância (fundo) com a cor verde usando o 'alpha'
    # Canal 0 (Red)   - Diminui a intensidade vermelha
    overlay_rgb[tumor_pixels, 0] = overlay_rgb[tumor_pixels, 0] * (1 - alpha)
    # Canal 1 (Green) - Adiciona o verde misturado com o tom de cinza
    overlay_rgb[tumor_pixels, 1] = overlay_rgb[tumor_pixels, 1] * (1 - alpha) + alpha
    # Canal 2 (Blue)  - Diminui a intensidade azul
    overlay_rgb[tumor_pixels, 2] = overlay_rgb[tumor_pixels, 2] * (1 - alpha)
    
    # Garante que os valores de cor não passem de 1.0 para o matplotlib não quebrar
    overlay_rgb = np.clip(overlay_rgb, 0, 1)

    # ────────────────────────────────────────────────────────────────────────
    # 6. PLOTAGEM
    # ────────────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))

    axes[0].imshow(image_np, cmap=cmap_img)
    axes[0].set_title(f"Ressonância Original (idx={idx})")

    axes[1].imshow(label_np, cmap="gray")
    axes[1].set_title("Máscara Real")

    # Exibe a imagem de pixel fundido (Ressonância com Tumor Verde)
    axes[2].imshow(overlay_rgb)
    axes[2].set_title("Predição (Ressonância + Tumor em Verde)")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(f"segmentacao_idx{idx}.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Imagem salva como segmentacao_idx{idx}.png")

# ─── USO ──────────────────────────────────────────────────────────────────────
# Troca test_loader.dataset pelo nome do seu dataset de teste
visualize_segmentation(model, test_loader.dataset, idx= random.randint(0, len(test_loader.dataset)-1)
, device=device)
print(f"Total de imagens: {len(test_loader.dataset)}")

# Para ver outras imagens, basta trocar o idx:
# visualize_segmentation(model, test_loader.dataset, idx=42, device=device)
# visualize_segmentation(model, test_loader.dataset, idx=100, device=device)
